# Left-Right Check

This notebook aims to certify that avgWTP_left and avgWTP_right are consistently defined through the pipelines. Points of concern are within:

1. Empirical distributinos
2. Generate fixations
3. Drift function

## Checking Empirical Distributions

In [12]:
# %load -r 236-242 simulation.py
if legend is None:
    legend = {"left": {1}, "right": {2}, "transition": {0}, "blank_fixation": {3}}
Lset = set(legend.get("left", set()))
Rset = set(legend.get("right", set()))
Tset = set(legend.get("transition", {0}))   # codes that behave like 0 (transitions)
Bset = set(legend.get("blank_fixation", {3}))   # codes that behave like 3 (do NOT count as transitions after first fix)
ignore_set = set(legend.get("ignore", set()))

The above is the default legend that is used in empirical distributions. Whenever any `empirical_distributions` object is generated, I typically define this in the cell just to make sure that the object is properly representing the fixations. Note that left is 1 and right is 2 here.

In [ ]:
# %load -r 257-258,315,327-330,379 simulation.py
def closest_bin(x, bins):
    return min(bins, key=lambda b: abs(x - b))

# ...

for row in trial_iter:
    # ...
    v_left  = float(get_vL(row))
    v_right = float(get_vR(row))
    sv_lookup = {"L": v_left - v_right, "R": v_right - v_left}

    # ...
    b = closest_bin(sv_lookup[lab], valueDiffs)

Here we see left relative values as being left - right, and right being right - left. In practice the closest_bin function should never have to rely on a close bin if valueDiffs are properly defined and passed to the function. In my exploratory notebooks, this is the case.

Since everything from empirical distributions was already verified in eum_empirical_distributions_check.ipynb, we must take the above as factual, and following steps should remain consistent.

## Checking Generate Fixations

In [22]:
# %load -r 407 simulation.py
def generate_fixations(dt, relative_value_difference, empirical_distributions, max_duration_s=30.0, rng=None):

When calling generate fixation, the relative value difference is standardly defined as avgWTP_left - avgWTP_right. The following logic should be consistent with it as the direction of the first fixation is assigned.

In [ ]:
# %load -r 461-462,465 simulation.py
probLeftFirst = float(empirical_distributions['probFixLeftFirst'])
left_first = rng.random() < probLeftFirst

rel_val_for_first = relative_value_difference if left_first else -relative_value_difference

Above, we see left_first being assigned through the empirical distributions object passed to the function. Additionally, the relative value for the first fixation is swapped depending on the direction of the first fixation. If it is left, then nothing changes and the first relative value is the relative value difference passed to it: avgWTP_left - avgWTP_right. If it is right, then the difference is made negative.

In [ ]:
# %load -r 477,482-483,493,510 simulation.py
code = 1 if left_first else 2

# ...
# After processing the first fixation to prepare for the second fixation
current_side_code = 2 if code == 1 else 1
abs_val = abs(relative_value_difference)

# ...
while global_time < max_duration_s:
    # After a fixation is processed, switch sides
    current_side_code = 2 if current_side_code == 1 else 1

In order, the code above assigns a code, 1 being left, and 2 being right, to the first fixation conditional on left_first. Then, the code swaps to represent the subject looking at the opposite stimulus. Continuously for the next 30 seconds, switching occurs. Importantly, fixation durations are pooled for both relative value differences (positive and negative).

In [ ]:
# %load -r 450-459,489,493,503 simulation.py
def pooled_fixation_durations(fixation_dict, abs_val):
    pool = []
    # exact match
    if abs_val in fixation_dict: pool.extend(fixation_dict[abs_val])
    if -abs_val in fixation_dict: pool.extend(fixation_dict[-abs_val])
    # nearest-key fallback if pool empty
    if not len(pool):
        k1 = min(fixation_dict.keys(), key=lambda kk: abs(abs(kk) - abs_val))
        pool.extend(fixation_dict[k1])
    return np.asarray(pool)

# ...

pooled2 = pooled_fixation_durations(fix2, abs_val)

# ...
# After processing the first fixation to prepare for the second fixation
while global_time < max_duration_s:

    # ...
    # For each fixation
    fdur = rng.choice(pooled2)

To be more consistent with the theoretical approach, I should reconstruct using a pooling of near bins instead of the opposite bin. Before doing so, I should check with empirical distributions to see the average duration within near bins in the middle fixations.

## Checking Drift Function

In [33]:
# %load -r 10-20 simulation.py
def drift_function(x, t, avgWTP_left, avgWTP_right, fixation):
    fixation_index = min(int(t/dt), len(fixation)-1)
    current_fixation = fixation[fixation_index]
    if current_fixation == 0: # saccade
        drift_val = 0
    elif current_fixation == 1: # left
        drift_val = drift * (avgWTP_left - avgWTP_right * theta)
    else: # right
        drift_val = drift * (avgWTP_left * theta - avgWTP_right)
    
    return np.ones_like(x) * drift_val

Although I custom define drift functions and models in a grid search due to the specific needs of PyDDM's parameter system, this is the default drift function that I use. We see that when a fixation is 1, this represents left, just like the empirical distributions legend. When the fixation is 2, this represents right. The only difference from this drift function to the legend is that blank fixations are removed and replaced with transitions. The motivating theory behind this is that the blank fixation-transition distinction was used during the experiment to track when participants would slightly deviate from a stimulus towards the center, but not crossing over to the other side through the middle of the two objects. If a blank fixation is not reported, then the time spent looking at the stimulus and darting away from it is combined with the time spent looking back at the stimulus a second time as one time block. Careful attention to the reported data shows that not all such activity was cleaned because there are instances of a participant looking back at the same object after making a blank fixation. However, for the intent of the aDDM simulation, this need not be considered.

Through an audit of existing code, I have confirmed that the left/right assignment is done correctly according to empirical distributions. However, the pooled fixations used in generate fixation may pose an issue. Instead, I will try to implement a fix by interpolating from the nearest bins.

## Fixing Pooling in Generate Fixations